<a href="https://colab.research.google.com/github/ug-hemanth/BIS-LAB/blob/main/Public%20Transpot%20using%20PSO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import random

# --- CONFIGURATION & CONSTANTS ---
SWARM_SIZE = 20
GENERATIONS = 50

# PSO Hyperparameters
W = 0.5   # Inertia weight (how much velocity to retain)
C1 = 1.5  # Cognitive coefficient (pull toward personal best)
C2 = 1.5  # Social coefficient (pull toward global best)

# Constraints for bus intervals (headways) in minutes
MIN_INTERVAL = 5.0
MAX_INTERVAL = 60.0

# Relative passenger demand for 3 time blocks: Morning, Afternoon, Evening
DEMAND = [1.5, 0.5, 1.2]

def fitness_function(timetable):
    """
    Evaluates a timetable. Lower score (total cost) is better.
    """
    passenger_wait_cost = 0
    operator_cost = 0

    for headway, demand in zip(timetable, DEMAND):
        # Shorter headways = lower passenger wait times
        passenger_wait_cost += (headway / 2) * demand

        # Shorter headways = more bus trips = higher operating cost
        operator_cost += (60 / headway) * 50

    return passenger_wait_cost + operator_cost

# --- INITIALIZATION ---
# Positions: Random intervals between 5 and 60 minutes
positions = [[random.uniform(MIN_INTERVAL, MAX_INTERVAL) for _ in range(3)] for _ in range(SWARM_SIZE)]
# Velocities: Initialized to zero
velocities = [[0.0 for _ in range(3)] for _ in range(SWARM_SIZE)]

# Tracks individual memory and swarm memory
pbest = list(positions)
pbest_fitness = [fitness_function(p) for p in pbest]

gbest = min(pbest, key=fitness_function)
gbest_fitness = fitness_function(gbest)

# --- MAIN PSO LOOP ---
for gen in range(GENERATIONS):
    for i in range(SWARM_SIZE):
        for d in range(3):  # Loop through the 3 time blocks (dimensions)
            r1, r2 = random.random(), random.random()

            # 1. Update Velocity vector
            cognitive = C1 * r1 * (pbest[i][d] - positions[i][d])
            social = C2 * r2 * (gbest[d] - positions[i][d])
            velocities[i][d] = W * velocities[i][d] + cognitive + social

            # 2. Update Position
            positions[i][d] += velocities[i][d]

            # 3. Boundary Control (Clamp values to legal bounds)
            if positions[i][d] < MIN_INTERVAL:
                positions[i][d] = MIN_INTERVAL
                velocities[i][d] = 0.0  # Reset speed if it hits the wall
            elif positions[i][d] > MAX_INTERVAL:
                positions[i][d] = MAX_INTERVAL
                velocities[i][d] = 0.0

        # 4. Evaluate Fitness & Update Memories
        current_fitness = fitness_function(positions[i])

        # Update Personal Best
        if current_fitness < pbest_fitness[i]:
            pbest[i] = list(positions[i])
            pbest_fitness[i] = current_fitness

            # Update Global Best
            if current_fitness < gbest_fitness:
                gbest = list(positions[i])
                gbest_fitness = current_fitness

# --- RESULTS ---
optimized_intervals = [round(interval, 1) for interval in gbest]
print(f"Optimized Headways (Morning, Afternoon, Evening): {optimized_intervals} minutes")
print(f"Minimized Cost Score: {gbest_fitness:.2f}")


Optimized Headways (Morning, Afternoon, Evening): [60.0, 60.0, 60.0] minutes
Minimized Cost Score: 246.00
